In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RCHBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        
        # Block1 - Depthwise Separable Conv
        self.depthwise1 = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size, stride=1, padding=kernel_size//2, groups=in_channels)
        self.pointwise1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        
        # Block2 - Dilated Convolution
        self.conv2 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=1, padding=3, dilation=2)
        
        # Block3 - Conv1x1 for Residual Shortcut
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0)
        
        # Group Normalization
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        
        # SE Attention
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, out_channels // 16, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels // 16, out_channels, kernel_size=1),
            nn.Sigmoid()
        )
        
        # DropBlock instead of Dropout
        self.dropblock = nn.Dropout2d(p=0.3)
        
        # Final Conv1x1 Residual Shortcut
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        # Block1: Depthwise Separable Conv
        x1 = F.relu(self.pointwise1(self.depthwise1(x)))
        
        # Block2: Dilated Conv
        x2 = F.relu(self.conv2(x))
        
        # Block3: Residual Conv1x1
        x3 = self.conv3(x1)
        
        # Normalization
        x1, x2 = self.gn1(x1), self.gn2(x2)
        
        # Combine the two parallel paths
        out = x1 + x2
        
        # Apply SE Attention
        out = out * self.se(out)
        
        # DropBlock Regularization
        out = self.dropblock(out)
        
        # Residual Connection with Shortcut
        out += self.shortcut(x)
        
        # Sigmoid Activation
        out = torch.sigmoid(out)
        
        return out


In [5]:
# RCH Block
class RCHBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size1=15, kernel_size2=7, final_layer=False):
        super().__init__()
        padding1 = (kernel_size1 - 1) // 2  
        padding2 = (kernel_size2 - 1) // 2  
        
        # Parallel Convolution Blocks
        self.block1 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size1, stride=1, padding=padding1)
        self.block2 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size1, stride=1, padding=padding1)
        
        # Additional 1x1 Convolution
        self.block3 = nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0)
        
        # Batch Normalization
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Sigmoid Activation
        self.final_layer = final_layer

    def forward(self, x):
        x1 = self.block1(x)
        x2 = self.block2(x)
        
        # Ensure x1 and x2 are the same shape before addition
        if x1.shape != x2.shape:
            x2 = F.interpolate(x2, size=x1.shape[2:], mode='bilinear', align_corners=False)

        # Pass x1 through additional conv block
        x3 = self.block3(x1)

        # Normalize
        x1, x3 = self.bn1(x1), self.bn2(x3)

        # Add both paths
        out = x1 + x3

        # Sigmoid Activation
        out = torch.sigmoid(out)

        return out

# RCH Classifier
class RCHClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        
        self.rch1 = RCHBlock(in_channels=3, out_channels=64, kernel_size1=7)
        self.pool1 = nn.AvgPool2d(2)  
        
        self.rch2 = RCHBlock(in_channels=64, out_channels=128, kernel_size1=5)
        self.pool2 = nn.AvgPool2d(2)  

        self.rch3 = RCHBlock(in_channels=128, out_channels=256, kernel_size1=3)
        self.pool3 = nn.AdaptiveAvgPool2d(1)  

        self.fc = nn.Linear(256, num_classes)  

    def forward(self, x):
        x = self.pool1(self.rch1(x)) 
        x = self.pool2(self.rch2(x)) 
        x = self.pool3(self.rch3(x))  

        x = torch.flatten(x, 1) 
        x = self.fc(x) 
        
        return x
# Model Summary
from torchsummary import summary

device = "cuda" if torch.cuda.is_available() else "cpu"
model = RCHClassifier(num_classes=2).to(device)

summary(model, (3, 256, 256))     

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 256, 256]           9,472
            Conv2d-2         [-1, 64, 256, 256]           9,472
            Conv2d-3         [-1, 64, 256, 256]           4,160
       BatchNorm2d-4         [-1, 64, 256, 256]             128
       BatchNorm2d-5         [-1, 64, 256, 256]             128
          RCHBlock-6         [-1, 64, 256, 256]               0
         AvgPool2d-7         [-1, 64, 128, 128]               0
            Conv2d-8        [-1, 128, 128, 128]         204,928
            Conv2d-9        [-1, 128, 128, 128]         204,928
           Conv2d-10        [-1, 128, 128, 128]          16,512
      BatchNorm2d-11        [-1, 128, 128, 128]             256
      BatchNorm2d-12        [-1, 128, 128, 128]             256
         RCHBlock-13        [-1, 128, 128, 128]               0
        AvgPool2d-14          [-1, 128,